**Richard Davies, Finn McEvoy, Josh Hellings** - Automated Data Visualisation for Policymaking 2026/27

In this notebook, we're going to produce a chart using two series from the Economics Observatory API that we'll transform with `Pandas` and present with `Vega-lite` (`Altair`). We will produce a chart showing Indian Exports and Imports as a percentage of GDP using data sourced from the API.

</br></br></br></br>

## Introducing Tools: `Requests`

The `Requests` module allows us to fetch resources from the internet, whether these are `CSVs`, `JSONs`, images, `HTML` or anything else. This is particuarly important for requesting data from APIs.

</br>

It is simple to use. Usually allow we need to do is:

1. Make a request with `requests.get` and our target URL. For example we can request something from our GitHub repos:

    `req = requests.get("https://raw.githubusercontent.com/mclass-user/mclass-user.github.io/main/s2_chart1.json")`
</br>
</br>
2. Access the fetched data. Using `req.json()` for JSON data or `req.text` for most other data. For example, we can see the returned JSON for the chart we just fetched:
    </br>
    </br>
    `data = req.json()`

First, let's import the modules we'll be using.

In [ ]:
## // Import the requests module
import requests

## // Import the pandas module.
import pandas as pd

In [ ]:
response = requests.get("https://raw.githubusercontent.com/RDeconomist/RDeconomist.github.io/refs/heads/main/charts/library/chartLine3.json")

# Let's try and view the data
print(response.json())

So we've managed to bring JSON data from online into our Python Notebook using the `requests` module.

</br></br>
</br></br>
</br>
</br>
</br>

# Transforming Data: Indian Exports and Imports

We want to produce a chart showing Indian exports and Imports as a percentage of GDP. To do this, we can use three series from the [Economics Observatory API](https://www.economicsobservatory.com/data-hub):
1. Indian Exports
2. Indian Imports
3. Indian GDP

</br>
</br>

We can find the API urls for these from the [Data-Hub](https://www.economicsobservatory.com/data-hub)

<img
style="max-height: 350px;
    width: auto;" src="https://raw.githubusercontent.com/jhellingsdata/RADataHub/main/misc/Masterclass/section%205/images/data-hub.png">
</img>

</br>
</br>

We find:

1. **Exports**: https://api.economicsobservatory.com/ind/expo
2. **Imports**: https://api.economicsobservatory.com/ind/impo
3. **GDP**: https://api.economicsobservatory.com/ind/gdpa

</br>
</br>


Let's try requesting the first of these, `Exports` and see how it's formatted.

In [ ]:
exports_req = requests.get("https://api.economicsobservatory.com/ind/expo")
exports_req.json()

</br>
</br>

This response contains some metadata, then the actual data is nested in the `data` key. (*Recall we looked at accessing nested dictionaries in the **PythonBasics** workbook.*)

Since we just care about the data itself, let's select just what's inside this `data` key and then make a dataframe out of it.

In [ ]:
exports_response = exports_req.json()
exports_data = exports_response['data']
exports_df = pd.DataFrame(exports_data)

exports_df.head(5)

We'll rename the `value` column to `export` so we can keep track of what this data is when we merge it.

In [ ]:
exports_df = exports_df.rename(columns={"value": "exports"})
exports_df.head(5)

</br>
</br>
</br>

We now have a Pandas `DataFrame` containing the exports data. Let's do the same for imports and GDP.

In [ ]:
imports_url = "https://api.economicsobservatory.com/ind/impo"
imports_req = requests.get(imports_url) # This is the request
imports_response = imports_req.json() # This is the response in JSON format
imports_data = imports_response['data'] # We only want the data part of the response, not the metadata
imports_df = pd.DataFrame(imports_data) # We convert the data to a DataFrame
imports_df = imports_df.rename(columns={"value": "imports"}) # We rename the value column to imports

gdp_url = "https://api.economicsobservatory.com/ind/gdpa"
gdp_req = requests.get(gdp_url)
gdp_response = gdp_req.json()
gdp_data = gdp_response['data']
gdp_df = pd.DataFrame(gdp_data)
gdp_df = gdp_df.rename(columns={"value": "gdp"})

In [ ]:
imports_df.head(2)

In [ ]:
gdp_df.head(2)

</br>
</br>

Now that we have all three series downloaded, we can start merging and transforming the data.

Let's merge our `gdp_df` with the `exports_df` and `imports_df` so we can express them as a ratio.

In [ ]:
exports_ratio_df = exports_df.merge(gdp_df, on="date")
exports_ratio_df.tail(5)

</br>
</br>

Let's now divide the `exports` column by the `gdp` column to get the Ratio and keep only the result and the `date`.

In [ ]:
exports_ratio_df['Exports Ratio'] = exports_ratio_df['exports'] / exports_ratio_df['gdp'] # Divide exports by GDP to get the ratio
exports_ratio_df = exports_ratio_df[['date', 'Exports Ratio']] # Select only the date and Exports Ratio columns
exports_ratio_df.head(5)

</br>
</br>
</br>

We'll also do the same to the `imports_df`:

In [ ]:
imports_ratio_df = imports_df.merge(gdp_df, on="date") # Merge imports and GDP DataFrames on the date column
imports_ratio_df['Imports Ratio'] = imports_ratio_df['imports'] / imports_ratio_df['gdp'] # Divide imports by GDP to get the ratio
imports_ratio_df = imports_ratio_df[['date', 'Imports Ratio']] # Select only the date and Imports Ratio columns
imports_ratio_df.head(5)

<br/><br/>

We now have all the data we need in two dataframes: `exports_ratio_df` and `imports_ratio_df`.

To use these in a graph, we'll need them in one dataframe.

In [ ]:
trade_df = pd.merge(exports_ratio_df, imports_ratio_df, on="date") # Merge the exports and imports DataFrames on the date column
trade_df.head(5)

</br></br>

It's best practice for `Vega-lite` to keep data in a TIDY format where there are columns that specify the series and a column for the data. To tranform our data, we'll use the pandad function `melt`.

In [ ]:
trade_df = trade_df.melt(id_vars="date", var_name="series", value_name="value") # Melt the DataFrame so that the exports and imports ratios are in the same column,
                                                                                # Specifying we want columns of date and series
trade_df

</br>
</br>

We can now export this data to use in `Vega-lite`.

In [ ]:
trade_df.to_csv("indian_trade_data.csv", index=False)

</br></br>

Finally, to get the CSV onto GitHub from Colab. We just have to go to the Files tab in the sidebar, and find our file:

<img
style="max-height: 350px;
    width: auto;" src="https://raw.githubusercontent.com/jhellingsdata/RADataHub/main/misc/Masterclass/section%205/images/colab_download.png">
</img>

Hit download, then you can add this to your VSCode and upload to GitHub.

---

### OPTIONAL

#### Charting with Altair

In [ ]:
## // Import the altair module
import altair as alt

Inspect our dataframe again.

In [ ]:
trade_df

So we have our data in nice **TIDY** format, allowing us to easily create a multi-line chart by specifying a third encoding channel, by which to separate the lines. We want separate lines by the `series` column, with distinguishing colours, so we can add a `color` encoding with `alt.Color("series:N")`.

In [ ]:
alt.Chart(trade_df).mark_line().encode(
    x=alt.X("date:T"),
    y=alt.Y("value:Q"),
    color=alt.Color("series:N")         ### We map the `series` column to the `color` encoding channel.
)